# 🛰️ SatQuery AI — Session 0: Drive Setup
**Run this ONCE before any training session.**

This notebook:
1. ✅ Verifies T4 GPU is available
2. ✅ Mounts Google Drive
3. ✅ Installs all dependencies
4. ✅ Clones SIH repo from GitHub
5. ✅ Creates Drive folder structure
6. ✅ Logs into HuggingFace
7. ✅ Downloads all datasets (~860 MB)
8. ✅ Tests model loading (sanity check)

In [1]:
# ── CELL 1: Verify T4 GPU ─────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip())

import torch
print(f'CUDA: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

if 'T4' not in result.stdout and 'A100' not in result.stdout:
    print('⚠️  Not a T4! Go to Runtime > Change runtime type > T4 GPU')
else:
    print('✅ T4 GPU confirmed!')

GPU: Tesla T4, 15360 MiB
CUDA: True
Device: Tesla T4
VRAM: 15.6 GB
✅ T4 GPU confirmed!


In [ ]:
# ── CELL 2: Mount Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/SatQuery_AI'
import os
folders = [
    f'{DRIVE}/ckpt/r1_warmup',
    f'{DRIVE}/ckpt/r2_binary_vqa/best',
    f'{DRIVE}/ckpt/r3a_mcq/best',
    f'{DRIVE}/ckpt/r3b_captioning/best',
    f'{DRIVE}/ckpt/r4_grounding/best',
    f'{DRIVE}/ckpt/r5_change/best',
    f'{DRIVE}/ckpt/r6_fusion/best',
    f'{DRIVE}/ckpt/merged',
    f'{DRIVE}/datasets',
    f'{DRIVE}/results',
    f'{DRIVE}/exported_model',
]
for f in folders:
    os.makedirs(f, exist_ok=True)
print('✅ Drive folders created!')
print(f'All checkpoints will be saved to: {DRIVE}')

In [ ]:
# ── CELL 3: Install dependencies ──────────────────────────────────────────
# Installs pre-built wheels to avoid rust compilation errors
!pip install -q --upgrade pip
!pip install -q transformers peft accelerate bitsandbytes datasets einops timm sentencepiece huggingface_hub
!pip install -q rouge-score nltk pycocotools rasterio opencv-python-headless jsonlines tensorboard

print('✅ All packages installed!')

In [ ]:
# ── CELL 4: Clone SIH repo ────────────────────────────────────────────────
import os

GITHUB_USER = 'abhineet115'  # ← your GitHub username
REPO = 'SIH'

if not os.path.exists(f'/content/{REPO}'):
    !git clone https://github.com/{GITHUB_USER}/{REPO}.git /content/{REPO}
else:
    !git -C /content/{REPO} pull origin main

os.chdir(f'/content/{REPO}/training')
print(f'✅ Working directory: {os.getcwd()}')
print('Files:', os.listdir('.'))

In [ ]:
# ── CELL 5: HuggingFace Login ─────────────────────────────────────────────
# Get your FREE token at: https://huggingface.co/settings/tokens
# Permissions needed: READ

from huggingface_hub import login
HF_TOKEN = 'hf_XXXXXXXXXXXXXXXXXXXX'  # ← paste your token here

login(token=HF_TOKEN)
print('✅ Logged into HuggingFace!')

In [ ]:
# ── CELL 6: Download all datasets (~860 MB) ───────────────────────────────
import json, requests
from pathlib import Path

DATASETS_DIR = f'{DRIVE}/datasets'

def download_file(url, dest):
    if Path(dest).exists():
        print(f'  Already exists: {dest}')
        return
    print(f'  Downloading {url} ...')
    r = requests.get(url, stream=True)
    with open(dest, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f'  ✅ Saved: {dest}')

print('Downloading RSVQA-LR (~8 MB)...')
!python data/download_all.py --output {DATASETS_DIR} --dataset rsvqa_lr

print('\nDownloading BEN-Bench (~500 MB)...')
!python data/download_all.py --output {DATASETS_DIR} --dataset ben_bench

print('\nDownloading CDVQA (~150 MB)...')
!python data/download_all.py --output {DATASETS_DIR} --dataset cdvqa

print('\nDownloading LEVIR-CD (~200 MB)...')
!python data/download_all.py --output {DATASETS_DIR} --dataset levir_cd

print('\n✅ All datasets downloaded!')
# Verify
!du -sh {DATASETS_DIR}/*

In [ ]:
# ── CELL 7: Sanity check — verify model can be loaded ─────────────────────
print('Testing InternVL3-1B loading (4-bit)...')
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

tok = AutoTokenizer.from_pretrained('OpenGVLab/InternVL3-1B', trust_remote_code=True)
m = AutoModelForCausalLM.from_pretrained(
    'OpenGVLab/InternVL3-1B',
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True,
)

# Check VRAM
vram_used = torch.cuda.memory_allocated() / 1e9
print(f'\n✅ Model loaded!')
print(f'VRAM used: {vram_used:.2f} GB / 15 GB')
print(f'Remaining VRAM: {15 - vram_used:.2f} GB (for activations + adapters)')

del m
torch.cuda.empty_cache()
print('\n✅ Setup complete! You are ready to start training.')
print('Next: Open 02_r1_warmup.ipynb')